# 0 Imports

In [31]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd

from src.suporte import funcoes_suporte as fs

## 0.1 Funções Suporte

In [32]:
fs.jupyter_settings(altura = 10, largura = 12, fonte = 8)
fs.supressao_notacao(casa_decimal = 2)

# 0.2 Load Data

In [33]:
df_fe = fs.load_pickle("../data/interim/1.descricao.pkl")
df_fe.sample(5)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
266534,560258,22847,BREAD BIN DINER STYLE IVORY,4,2011-07-17,14.95,13340,United Kingdom
397323,571180,20961,STRAWBERRY BATH SPONGE,20,2011-10-14,1.25,15498,United Kingdom
389381,570467,22204,MILK PAN BLUE POLKADOT,4,2011-10-10,3.75,12607,USA
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09,2.08,16446,United Kingdom
291814,562519,85032A,ROMANTIC IMAGES GIFT WRAP SET,4,2011-08-05,0.65,16764,United Kingdom


# 1.0 F.E.

In [34]:
df_ref = df_fe['customer_id'].drop_duplicates( ignore_index=True)
df_ref.head()

0    17850
1    13047
2    12583
3    13748
4    15100
Name: customer_id, dtype: int64

## 1.1 Gross Revenue (Faturamento) quantidade * preço

In [35]:
df_fe["faturamento"] = df_fe["quantity"] * df_fe["unit_price"]

## 1.2 Monetário

In [36]:
df_monetario = df_fe[["customer_id", "faturamento"]].groupby("customer_id").sum().reset_index()

df_ref = pd.merge(df_ref, df_monetario, how="left", on="customer_id")

df_ref.sample()

,customer_id,faturamento
720,17969,228.20


In [37]:
df_ref.isna().sum()

customer_id    0
faturamento    0
dtype: int64

In [43]:
del df_monetario

## 1.3 Recência

In [44]:
df_recencia = df_fe[["customer_id", "invoice_date"]].groupby("customer_id").max().reset_index()

df_recencia["recencia_days"] = (df_fe["invoice_date"].max() - df_recencia["invoice_date"]).dt.days

df_recencia = df_recencia[["customer_id", "recencia_days"]].copy()

df_ref = pd.merge(df_ref, df_recencia, how = "left", on="customer_id")

df_ref.head()

,customer_id,faturamento,recencia_days
0,17850,5288.63,302
1,13047,3079.10,31
2,12583,7187.34,2
3,13748,948.25,95
4,15100,635.10,330


In [45]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
dtype: int64

In [46]:
del df_recencia

## 1.4 Frequência

In [51]:
df_frequencia = df_fe[["customer_id", "invoice_no"]].drop_duplicates().groupby("customer_id").count().reset_index()

df_ref = pd.merge(df_ref, df_frequencia, how='left', on='customer_id')
df_ref.head()

,customer_id,faturamento,recencia_days,invoice_no
0,17850,5288.63,302,35
1,13047,3079.10,31,18
2,12583,7187.34,2,18
3,13748,948.25,95,5
4,15100,635.10,330,6


In [52]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
invoice_no       0
dtype: int64

In [53]:
del df_frequencia

# 2.0 Exportar DF

In [54]:
path = "../data/interim/2.fe.pkl"
fs.save_pickle(obj=df_ref,path=path)